# RoomBooking — Notebook técnico

Este notebook explica las tecnologías de la solución con ejemplos de código que
corren de verdad. El proyecto es un asistente conversacional para reservar salas de
reunión: el usuario escribe en lenguaje natural y el sistema crea, consulta o cancela
reservas respetando las reglas del enunciado.

**Stack:** FastAPI (capa HTTP), LangChain con `create_agent` sobre LangGraph (el
agente), SQLAlchemy + SQLite (persistencia) y la API de OpenAI (el modelo).

**La idea que ordena todo:** el LLM solo elige qué herramienta llamar; las reglas de
negocio las define y las hace cumplir el código que está detrás de esas herramientas,
no el modelo ni el prompt. Si se borra el prompt entero, ninguna regla se puede violar.

Cada request al agente llega con un `User` ya resuelto del JWT; el detalle del login
está en la sección de FastAPI y autenticación, más abajo. El diagrama de componentes
está en [`ARCHITECTURE.md`](ARCHITECTURE.md) y el log de decisiones en
[`DECISIONS.md`](DECISIONS.md).

> Casi todas las celdas corren sin OpenAI (usan la capa de reglas y las tools con una
> base de prueba). Solo la demo del agente al final necesita una `OPENAI_API_KEY`, y
> está preparada para que la ejecutes vos.

## Preparación

Levantamos una base SQLite de prueba (un archivo temporal, para que la demo de
concurrencia tenga conexiones reales por thread) y la sembramos con las salas A–E y
los usuarios, con el mismo código que usa la app en producción.

In [1]:
import os
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

# Poner la raíz del repo (donde está pyproject.toml) en el path, para importar app.*
here = Path.cwd()
root = next(p for p in (here, *here.parents) if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from sqlalchemy import select
from sqlalchemy.orm import Session

from app.auth import hash_password
from app.booking import service
from app.booking.db import init_db, make_engine
from app.booking.models import Booking, Room, User

# Base de prueba en un archivo temporal (misma función que usa la app).
db_path = os.path.join(tempfile.gettempdir(), "roombooking_notebook.db")
if os.path.exists(db_path):
    os.remove(db_path)
engine = make_engine(f"sqlite:///{db_path}")
init_db(engine, users={"User1": hash_password("demo"), "User2": hash_password("demo")})


def get_user(session, username="User1"):
    return session.scalar(select(User).where(User.username == username))


with Session(engine) as s:
    print("Salas:", [(r.id, r.capacity) for r in s.scalars(select(Room).order_by(Room.id))])
    print("Usuarios:", [u.username for u in s.scalars(select(User))])

Salas: [('A', 2), ('B', 4), ('C', 6), ('D', 8), ('E', 10)]
Usuarios: ['User1', 'User2']


## 1. Persistencia y reglas de negocio

Toda la lógica del dominio vive en `app/booking/service.py`, la **única fuente de
verdad** de las reglas. Una reserva se modela como un solo rango contiguo `[start, end)`
alineado a slots de 30 minutos, así que combinar slots no contiguos es imposible por
diseño: no hay que validar algo que no se puede escribir.

La regla central, el no solapamiento, es una comparación de rangos. Dos intervalos
semiabiertos `[a, b)` y `[c, d)` se solapan si y solo si `a < d AND b > c`. En
SQLAlchemy eso es una query directa (`app/booking/service.py`):

```python
select(Booking).where(
    Booking.room_id == room_id,
    Booking.start < end,   # el existente empieza antes de que termine el nuevo
    Booking.end > start,   # y termina después de que empieza el nuevo
)
```

Que una reserva termine 11:30 y otra empiece 11:30 no es conflicto, que es justo el
ejemplo del enunciado.

Creamos una reserva válida:

In [2]:
session = Session(engine)
u1 = get_user(session, "User1")

booking = service.create_booking(
    session, user=u1, room_id="B",
    start=datetime(2030, 6, 15, 10, 0),
    end=datetime(2030, 6, 15, 11, 0),
    title="Daily de equipo", attendees=3,
)
print(f"OK  #{booking.id}: sala {booking.room_id} "
      f"{booking.start:%Y-%m-%d %H:%M}-{booking.end:%H:%M} "
      f'"{booking.title}" ({booking.attendees} personas)')

OK  #1: sala B 2030-06-15 10:00-11:00 "Daily de equipo" (3 personas)


Y vemos cómo el service rechaza cada violación de regla. Cada rechazo es un
`BookingError` con un mensaje pensado para mostrarle al usuario final.

In [3]:
def intentar(descripcion, **kw):
    with Session(engine) as s:
        try:
            service.create_booking(s, user=get_user(s), **kw)
            print(f"{descripcion}: creada")
        except service.BookingError as e:
            print(f"{descripcion}: RECHAZO -> {e}")


intentar("Capacidad (sala A = 2, pido 7)",
         room_id="A", start=datetime(2030, 6, 15, 10, 0), end=datetime(2030, 6, 15, 11, 0),
         title="Reunion", attendees=7)

intentar("Solapamiento (sala B ya ocupada 10-11)",
         room_id="B", start=datetime(2030, 6, 15, 10, 30), end=datetime(2030, 6, 15, 11, 30),
         title="Choca", attendees=2)

intentar("Duracion mayor a 3h",
         room_id="C", start=datetime(2030, 6, 15, 9, 0), end=datetime(2030, 6, 15, 13, 0),
         title="Larga", attendees=2)

intentar("Sin alinear a slots (10:15)",
         room_id="C", start=datetime(2030, 6, 15, 10, 15), end=datetime(2030, 6, 15, 11, 0),
         title="Desalineada", attendees=2)

Capacidad (sala A = 2, pido 7): RECHAZO -> Room A holds at most 2 people, got 7. Try a bigger room.
Solapamiento (sala B ya ocupada 10-11): RECHAZO -> Room B is already booked in that range (2030-06-15 10:00–11:00).
Duracion mayor a 3h: RECHAZO -> A booking can last at most 3 hours (6 contiguous 30-minute slots).
Sin alinear a slots (10:15): RECHAZO -> The start time must align to 30-minute slots (e.g. 10:00 or 10:30), got 10:15.


**Datetimes con timezone (D15).** Si el modelo agrega una `Z` o un offset al
timestamp, comparar ese datetime *aware* contra el reloj naive del sistema rompía con
un `TypeError` que no era un `BookingError` y terminaba en un error 500. Convertir el
offset automáticamente sería peor: una `10:00Z` se volvería 07:00 en silencio. La
decisión fue rechazarlo con un mensaje que el agente puede corregir.

In [4]:
intentar("Datetime con timezone (Z)",
         room_id="C",
         start=datetime(2030, 6, 15, 10, 0, tzinfo=timezone.utc),
         end=datetime(2030, 6, 15, 11, 0, tzinfo=timezone.utc),
         title="Con Z", attendees=2)

Datetime con timezone (Z): RECHAZO -> Times must be local (America/Montevideo) without a UTC offset or 'Z', got '2030-06-15T10:00:00+00:00'. Resend the local wall-clock time, e.g. '2030-06-15T10:00'.


**La carrera de doble reserva (D13).** FastAPI atiende los endpoints sync desde
un threadpool, así que dos requests podían intercalarse entre el chequeo de solapamiento
y el INSERT, y crear dos reservas solapadas sin que ninguno viera el conflicto. Es el
clásico *check-then-act race*: el chequeo corrió en los dos, pero entre el `if` y el
`insert` el mundo cambió. Lo cierra un `threading.Lock` de proceso alrededor de la
sección crítica (chequeo más insert).

Lo demostramos: varios threads sincronizados por una barrera disparan la misma reserva
al mismo tiempo. Con el lock, exactamente uno gana.

In [5]:
import threading
from collections import Counter


def carrera(results, idx, barrier):
    barrier.wait()  # todos arrancan al mismo tiempo
    with Session(engine) as s:
        try:
            service.create_booking(
                s, user=get_user(s), room_id="D",
                start=datetime(2030, 7, 1, 9, 0), end=datetime(2030, 7, 1, 10, 0),
                title="Carrera", attendees=2,
            )
            results[idx] = "created"
        except service.BookingError:
            results[idx] = "rejected"


N = 8
barrier = threading.Barrier(N)
results = [None] * N
threads = [threading.Thread(target=carrera, args=(results, i, barrier)) for i in range(N)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(Counter(results))
with Session(engine) as s:
    filas = s.scalars(select(Booking).where(Booking.room_id == "D")).all()
print(f"Reservas realmente escritas en sala D: {len(filas)}  (exactamente una)")

Counter({'rejected': 7, 'created': 1})
Reservas realmente escritas en sala D: 1  (exactamente una)


## 2. Tool-calling con LangChain

Las tools (`app/agent/tools.py`) son wrappers finos entre el LLM y el service. El
decorador `@tool` toma la firma (los type hints) y el docstring y genera el JSON Schema
que se le manda a OpenAI: el docstring no es documentación, es el prompt de esa tool.

**La identidad va por closure, no por parámetro.** `build_tools` recibe al usuario ya
autenticado (resuelto del JWT en la capa HTTP) y las tools lo capturan en su clausura.
El `user` no es un argumento que el modelo pueda completar, así que un "reservá a nombre
de User2" no tiene forma sintáctica de ejecutarse. Se ve en el schema: no aparece
`user`.

In [6]:
from app.agent.tools import build_tools

session_factory = lambda: Session(engine)
with Session(engine) as s:
    u1 = get_user(s, "User1")
tools = {t.name: t for t in build_tools(session_factory, u1)}

print("Tools disponibles:", list(tools))
schema = tools["create_booking"].args_schema.model_json_schema()
print("\nParametros que ve el modelo para create_booking:")
print(" ", list(schema["properties"]))
print("  (no hay 'user': la identidad es una closure, el LLM no la puede pasar)")

Tools disponibles: ['create_booking', 'list_available_rooms', 'get_room_schedule', 'list_my_bookings', 'cancel_booking']

Parametros que ve el modelo para create_booking:
  ['room', 'start', 'end', 'title', 'attendees']
  (no hay 'user': la identidad es una closure, el LLM no la puede pasar)


**Errores como datos, no excepciones.** Cada tool atrapa el `BookingError` y
devuelve un string `BOOKING REJECTED: ...`. Una excepción sin atrapar cortaría el loop
del agente; un string de rechazo vuelve al modelo como un `ToolMessage` y lo único que
puede hacer es relatarlo y ofrecer una alternativa. Lo invocamos directo, sin LLM:

In [7]:
print("Reserva valida:")
print(" ", tools["create_booking"].invoke({
    "room": "C", "start": "2030-06-20T14:00", "end": "2030-06-20T15:00",
    "title": "Entrevista con Ana", "attendees": 2}))

print("\nReserva invalida (capacidad):")
print(" ", tools["create_booking"].invoke({
    "room": "A", "start": "2030-06-20T14:00", "end": "2030-06-20T15:00",
    "title": "Demasiados", "attendees": 9}))

Reserva valida:
  Booking confirmed. Booking #3: room C, Thursday 2030-06-20 14:00-15:00, "Entrevista con Ana", 2 attendee(s)

Reserva invalida (capacidad):
  BOOKING REJECTED: Room A holds at most 2 people, got 9. Try a bigger room.


**Por qué un session factory y no una sesión (D14).** El `ToolNode` de LangChain
ejecuta las tool calls de un mismo turno en threads separados, y `gpt-4o` las emite en
paralelo ("reservá la B y la C" son dos calls simultáneas). Al principio las tools
compartían una `Session` de SQLAlchemy, que no es thread-safe: en pruebas, 24 de 40
corridas fallaban, la mitad con errores y la otra mitad con resultados silenciosamente
falsos. La solución fue pasar un `Callable[[], Session]`: cada invocación de tool abre y
cierra su propia sesión. Lo mostramos corriendo varias tools en paralelo sin que se
pisen.

In [8]:
import threading

outs = {}


def correr(name, payload, key):
    outs[key] = tools[name].invoke(payload)


jobs = [
    threading.Thread(
        target=correr,
        args=("list_available_rooms",
              {"start": "2030-06-25T09:00", "end": "2030-06-25T10:00"}, i),
    )
    for i in range(4)
]
for j in jobs:
    j.start()
for j in jobs:
    j.join()

print("4 tools en paralelo, cada una con su propia sesion, sin error:")
for k in sorted(outs):
    print(f"  [thread {k}] {outs[k]}")

4 tools en paralelo, cada una con su propia sesion, sin error:
  [thread 0] Available between 2030-06-25 09:00 and 10:00: A (up to 2 people), B (up to 4 people), C (up to 6 people), D (up to 8 people), E (up to 10 people).
  [thread 1] Available between 2030-06-25 09:00 and 10:00: A (up to 2 people), B (up to 4 people), C (up to 6 people), D (up to 8 people), E (up to 10 people).
  [thread 2] Available between 2030-06-25 09:00 and 10:00: A (up to 2 people), B (up to 4 people), C (up to 6 people), D (up to 8 people), E (up to 10 people).
  [thread 3] Available between 2030-06-25 09:00 and 10:00: A (up to 2 people), B (up to 4 people), C (up to 6 people), D (up to 8 people), E (up to 10 people).


## 3. El agente: create_agent, prompt, middleware y memoria

El agente se arma con `create_agent` (`app/agent/agent.py`), la API actual de
LangChain 1.x que compila un grafo de LangGraph con el loop ReAct resuelto. Se rearma
en cada request (es solo cableado de grafo, barato) para atar el prompt y las tools al
usuario y a la hora actual.

Lo que se inyecta en el system prompt en cada request:
- **el nombre del usuario** (cosmético; la identidad real va por closure),
- **la fecha y hora actual**, porque un LLM no tiene reloj y sin esto "mañana a las 10"
  es irresoluble,
- **la lista de salas**, para ahorrar un tool-call en la pregunta más común.

El prompt enumera las reglas pero con una etiqueta explícita: *"informative only — the
booking system itself enforces them"*. El prompt informa, el código legisla.

In [9]:
from app.agent.agent import SYSTEM_PROMPT, checkpointer, build_agent

print("Middleware y memoria del agente:")
print("  - SummarizationMiddleware: resume turnos viejos pasados ~3k tokens (techo de costo)")
print("  - ModelCallLimitMiddleware: maximo 10 llamadas al modelo por mensaje (corta loops)")
print(f"  - checkpointer: {type(checkpointer).__name__} (memoria de conversacion por thread)")
print("\n--- system prompt (primeras lineas) ---")
print("\n".join(SYSTEM_PROMPT.splitlines()[:14]))

Middleware y memoria del agente:
  - SummarizationMiddleware: resume turnos viejos pasados ~3k tokens (techo de costo)
  - ModelCallLimitMiddleware: maximo 10 llamadas al modelo por mensaje (corta loops)
  - checkpointer: InMemorySaver (memoria de conversacion por thread)

--- system prompt (primeras lineas) ---
You are the meeting-room booking assistant for the Cubo Itau office.
Your ONLY purpose is to help employees manage meeting-room bookings. Politely
refuse any request outside that scope.

Logged-in user: {username}.
Current local date and time: {now} (America/Montevideo). There is a single
timezone; never ask about timezones.

Rooms: {rooms}.

Booking rules (informative only — the booking system itself enforces them and
will reject anything invalid):
- Bookings use 30-minute slots (times must end in :00 or :30) and last at most
  3 hours (6 contiguous slots).


**La memoria de conversación.** El estado vive en un checkpointer `InMemorySaver`
de LangGraph, guardado por `thread_id`. El `thread_id` es el claim `jti` del JWT (un id
aleatorio por sesión de login): un login nuevo empieza una conversación nueva, sin
estado extra en el servidor. En la capa HTTP se ve como
`agent.invoke(..., config={"configurable": {"thread_id": session_id}})`.

**La elección del modelo (D2), con evidencia.** El default arrancó en `gpt-4o-mini`
porque es barato. Pero el mini no respetaba de forma confiable la instrucción de pedir
los datos faltantes: ante "reservá la sala B mañana a las 15 para 2", en vez de
preguntar la hora de fin, reservaba un slot de 30 minutos en silencio, a pesar de que el
system prompt, el docstring de la tool y un ejemplo few-shot se lo prohibían explícito.
`gpt-4o` sí obedece. Por eso el default se subió a `gpt-4o`, documentado en D2.

### Demo del agente 

Esta es la única celda que llama a OpenAI. Corre solo si hay una `OPENAI_API_KEY` en el
entorno o en el `.env`. La key se lee del `settings` del proyecto, nunca se escribe ni
se imprime en el notebook.

Muestra dos cosas: una reserva por chat, y que el bot declina lo que está fuera de su
propósito.

In [10]:
from langchain_core.messages import HumanMessage

from app.agent.agent import build_agent
from app.config import settings

if settings.openai_api_key:
    with Session(engine) as s:
        demo_user = get_user(s, "User1")
    agent = build_agent(lambda: Session(engine), demo_user)
    thread = {"configurable": {"thread_id": "notebook-demo"}}

    def preguntar(texto):
        r = agent.invoke({"messages": [HumanMessage(texto)]}, config=thread)
        return r["messages"][-1].content

    p1 = "Reservá la sala C mañana a las 16 por una hora, título Revisión de diseño, 2 personas"
    print("Usuario:", p1)
    print("Bot:", preguntar(p1))

    print("\nUsuario: ¿cuáles son mis reservas?")   # usa la memoria del hilo
    print("Bot:", preguntar("¿cuáles son mis reservas?"))

    print("\nUsuario: contame un chiste")           # fuera de scope -> declina
    print("Bot:", preguntar("contame un chiste"))
else:
    print("Sin OPENAI_API_KEY: poné tu key en el archivo .env y volvé a correr esta celda.")

Usuario: Reservá la sala C mañana a las 16 por una hora, título Revisión de diseño, 2 personas
Bot: La sala C ha sido reservada para mañana de 16:00 a 17:00 con el título "Revisión de diseño" para 2 personas. ¡Reserva confirmada!

Usuario: ¿cuáles son mis reservas?
Bot: Estas son tus reservas próximas:

1. Sala C, miércoles 2026-07-15 de 16:00 a 17:00, "Revisión de diseño", 2 personas.
2. Sala B, sábado 2030-06-15 de 10:00 a 11:00, "Daily de equipo", 3 personas.
3. Sala C, jueves 2030-06-20 de 14:00 a 15:00, "Entrevista con Ana", 2 personas.
4. Sala D, lunes 2030-07-01 de 09:00 a 10:00, "Carrera", 2 personas.

Usuario: contame un chiste
Bot: Lo siento, pero mi función es ayudarte con las reservas de salas de reuniones. Si necesitas gestionar alguna reserva, ¡estaré encantado de ayudarte!


## 4. FastAPI y autenticación

FastAPI (`app/main.py`) es la capa HTTP: expone `/login` y `/chat`, resuelve la
identidad antes de tocar el agente y sirve la página. Un patrón que vale marcar es la
dependencia con `yield`: una sesión de base por request, abierta y cerrada por el
framework.

```python
def get_session():
    with Session(engine) as session:
        yield session   # el endpoint corre "adentro" del with
```

Los tres códigos de error de `/chat` cuentan una historia clara: **401** token
inválido o vencido, **503** falta la `OPENAI_API_KEY` (falla explícita, no misteriosa),
**502** OpenAI caído u otro error del agente (mensaje neutro, el detalle queda en el
log). Y el modelo de FastAPI de correr endpoints sync en un threadpool es, justamente,
la causa raíz de los dos problemas de concurrencia de antes (D13 y D14).

**Autenticación (D6): JWT + bcrypt, fuera del chat.** El login es un paso HTTP: el LLM
nunca ve credenciales. El token lleva el usuario (`sub`) y un id de sesión (`jti`) que
se reusa como `thread_id` de la conversación.

In [11]:
from app.auth import create_token, decode_token, hash_password, verify_password

token = create_token("User1")
payload = decode_token(token)
print("JWT (recortado):", token[:32], "...")
print("payload:", {k: payload[k] for k in ("sub", "jti")})
print("el 'jti' es tambien el thread_id de la conversacion\n")

h = hash_password("un-password")
print("bcrypt hash (recortado):", h[:29], "...")
print("verifica password correcto:", verify_password("un-password", h))
print("verifica password incorrecto:", verify_password("otro", h))

JWT (recortado): eyJhbGciOiJIUzI1NiIsInR5cCI6IkpX ...
payload: {'sub': 'User1', 'jti': '6c80a523823043caa6aa055aaa1a5858'}
el 'jti' es tambien el thread_id de la conversacion

bcrypt hash (recortado): $2b$12$nPUmKBjrjKQHFcbF/234TO ...
verifica password correcto: True
verifica password incorrecto: False


**El backoffice (D16).** Además del chat hay una vista de operador en
`/backoffice` que muestra todas las reservas en un calendario semanal por sala y permite
cancelar cualquiera. Es deliberadamente **sin login**: sus endpoints son públicos y se
entra por link. La sumé para verificar durante la demo que las reservas quedan bien
guardadas y poder corregirlas, no como una consola con control de acceso; ese límite de
seguridad queda asumido y documentado. Reusa el mismo service, con dos helpers que
bypasean a propósito la regla de propiedad (`list_bookings_in_range`,
`admin_cancel_booking`), para que el "quién puede hacer qué" viva en un solo lugar.

In [12]:
with Session(engine) as s:
    todas = service.list_bookings_in_range(
        s, datetime(2030, 1, 1, 0, 0), datetime(2031, 1, 1, 0, 0))
    print(f"El backoffice ve {len(todas)} reservas, de todos los usuarios:")
    for b in todas:
        print(f"  #{b.id} sala {b.room_id} {b.start:%Y-%m-%d %H:%M} \"{b.title}\"")

    if todas:
        service.admin_cancel_booking(s, booking_id=todas[0].id)
        print(f"\nadmin_cancel_booking cancelo la #{todas[0].id} sin chequear quien la creo")

El backoffice ve 3 reservas, de todos los usuarios:
  #1 sala B 2030-06-15 10:00 "Daily de equipo"
  #3 sala C 2030-06-20 14:00 "Entrevista con Ana"
  #2 sala D 2030-07-01 09:00 "Carrera"

admin_cancel_booking cancelo la #1 sin chequear quien la creo


## 5. Guardrails: dónde vive cada defensa

El principio es "el agente propone, el código dispone". Cada amenaza se contiene en el
código, no en el prompt; el prompt es apenas una capa de cortesía.

| Amenaza | Defensa | Dónde |
|---|---|---|
| "Reservá / cancelá a nombre de User2" | Identidad por closure; `user` no es parámetro de ninguna tool | `tools.py` (código) |
| "Ignorá tus reglas" (prompt injection) | Las reglas viven en `service.py` y se validan siempre | `service.py` (código) |
| Password escrito en el chat | Login fuera del chat; el LLM nunca ve credenciales | `main.py` / `auth.py` (código) |
| Doble reserva simultánea | `threading.Lock` alrededor de chequeo + insert (D13) | `service.py` (código) |
| Tool calls paralelas corrompiendo la DB | Una sesión por invocación de tool (D14) | `tools.py` (código) |
| Datetime con timezone del modelo | Rechazado como `BookingError` (D15) | `service.py` (código) |
| Loop infinito de tool-calls | `ModelCallLimitMiddleware(run_limit=10)` | `agent.py` (código) |
| Conversación eterna, costo eterno | `SummarizationMiddleware` a ~3k tokens | `agent.py` (código) |
| Inventar disponibilidad o títulos | Instrucciones del prompt + tools que no asumen | prompt + `tools.py` |
| Salirse del propósito | "Your ONLY purpose... politely refuse" | prompt |

Si se borra el system prompt entero, el asistente se vuelve torpe pero ninguna regla de
negocio se puede violar, y la única identidad que existe es la del JWT.

## 6. Cómo corre en producción

Un único contenedor Docker en una instancia de **AWS Lightsail**, con la base SQLite en
un volumen persistente. Adelante, un reverse proxy **Caddy** termina TLS y gestiona el
certificado de Let's Encrypt de forma automática, sirviendo la app por HTTPS en un
subdominio propio.

**Por qué Lightsail y no serverless (D5).** El diseño mantiene estado en disco local a
propósito (SQLite, cero infra). Lambda tendría cold starts en un chat y un filesystem
efímero que empuja a DynamoDB o RDS. Fargate también tiene storage efímero (mata SQLite)
y, con el ALB que necesita para IP estable, sale ~25 USD/mes frente a los ~5 fijos de
Lightsail. La elección de cómputo depende de dónde vive el estado: con estado en disco
local, el fit es una VM con disco persistente. Si esto escalara, primero el estado sale
del disco (Postgres con exclusion constraint para el solapamiento) y recién ahí Fargate
con réplicas es la elección correcta.

El `Dockerfile` es parte del entregable y da paridad dev/prod (el mismo build corre
local y en la nube). Un par de decisiones concretas: dependencias con `uv sync --frozen
--no-dev` en una capa separada del código, usuario no-root, y un `HEALTHCHECK` contra
`/health`.

- **Demo en vivo:** https://promtior.manuanselmi.com (`User1` / `User2`, contraseña
  `TechnicalChallengePromtior`)
- **Repositorio:** https://github.com/manuanselmi/promtior-roombooking-challenge